# S07 — Solutions: Object-Oriented Programming

**Python for Applied Physics** | Master in Applied Physics  
⚠️ *Instructor reference — do not distribute before the exercise session.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py, tempfile, os
from dataclasses import dataclass
from typing import NamedTuple

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

---
## Exercise 1 — `GaussianBeam` class

In [ ]:
class GaussianBeam:
    c = 3e8

    def __init__(self, wavelength, w0, power):
        self.wavelength = wavelength
        self.w0         = w0
        self.power      = power

    @property
    def wavelength(self): return self._wavelength
    @wavelength.setter
    def wavelength(self, v):
        if v <= 0: raise ValueError(f"wavelength must be positive, got {v}")
        self._wavelength = float(v)

    @property
    def w0(self): return self._w0
    @w0.setter
    def w0(self, v):
        if v <= 0: raise ValueError(f"w0 must be positive, got {v}")
        self._w0 = float(v)

    @property
    def power(self): return self._power
    @power.setter
    def power(self, v):
        if v < 0: raise ValueError(f"power must be non-negative, got {v}")
        self._power = float(v)

    @property
    def rayleigh_range(self): return np.pi * self.w0**2 / self.wavelength

    @property
    def peak_intensity(self): return 2 * self.power / (np.pi * self.w0**2)

    @property
    def frequency(self): return self.c / self.wavelength

    def w(self, z): return self.w0 * np.sqrt(1 + (z/self.rayleigh_range)**2)

    def intensity(self, r, z):
        wz = self.w(z)
        return self.peak_intensity * (self.w0/wz)**2 * np.exp(-2*r**2/wz**2)

    def __repr__(self):
        return (f"GaussianBeam(λ={self.wavelength*1e9:.0f} nm, "
                f"w₀={self.w0*1e6:.0f} µm, P={self.power*1e3:.1f} mW)")


beam = GaussianBeam(wavelength=800e-9, w0=500e-6, power=1.0)
print(repr(beam))
assert beam.rayleigh_range > 0
assert np.isclose(beam.w(0), beam.w0)
assert np.isclose(beam.w(beam.rayleigh_range), beam.w0*np.sqrt(2), rtol=1e-6)
assert np.isclose(beam.intensity(np.array([0.0]), 0)[0], beam.peak_intensity, rtol=1e-6)
try: GaussianBeam(-1e-9, 500e-6, 1.0); assert False
except ValueError: pass
print(f"zR={beam.rayleigh_range*100:.2f} cm, I₀={beam.peak_intensity/1e4:.2f} W/cm²")
print("All assertions passed ✓")

---
## Exercise 2 — Property validation with M²

In [ ]:
class GaussianBeam:
    c = 3e8

    def __init__(self, wavelength, w0, power, M2=1.0):
        self.wavelength = wavelength; self.w0 = w0
        self.power = power;           self.M2 = M2

    @property
    def wavelength(self): return self._wavelength
    @wavelength.setter
    def wavelength(self, v):
        if v <= 0: raise ValueError(f"wavelength must be positive")
        self._wavelength = float(v)

    @property
    def w0(self): return self._w0
    @w0.setter
    def w0(self, v):
        if v <= 0: raise ValueError("w0 must be positive")
        self._w0 = float(v)

    @property
    def power(self): return self._power
    @power.setter
    def power(self, v):
        if v < 0: raise ValueError("power must be non-negative")
        self._power = float(v)

    @property
    def M2(self): return self._M2
    @M2.setter
    def M2(self, v):
        if v < 1.0: raise ValueError(f"M² must be ≥ 1, got {v}")
        self._M2 = float(v)

    @property
    def rayleigh_range(self): return np.pi * self.w0**2 / (self.M2 * self.wavelength)

    @property
    def divergence(self): return self.M2 * self.wavelength / (np.pi * self.w0)

    @property
    def peak_intensity(self): return 2 * self.power / (np.pi * self.w0**2)

    def w(self, z): return self.w0 * np.sqrt(1 + (z/self.rayleigh_range)**2)

    def intensity(self, r, z):
        wz = self.w(z)
        return self.peak_intensity * (self.w0/wz)**2 * np.exp(-2*r**2/wz**2)

    def __repr__(self):
        return (f"GaussianBeam(λ={self.wavelength*1e9:.0f} nm, "
                f"w₀={self.w0*1e6:.0f} µm, P={self.power*1e3:.1f} mW, M²={self.M2:.2f})")


beam1 = GaussianBeam(800e-9, 500e-6, 1.0, M2=1.0)
beam2 = GaussianBeam(800e-9, 500e-6, 1.0, M2=1.5)
assert beam1.rayleigh_range > beam2.rayleigh_range
assert np.isclose(beam1.divergence * beam1.w0, beam1.wavelength / np.pi, rtol=1e-6)
try: beam1.M2 = 0.5; assert False
except ValueError as e: print(f"Caught: {e}")
print(beam1); print(beam2)
print(f"zR (M²=1.0): {beam1.rayleigh_range*100:.2f} cm")
print(f"zR (M²=1.5): {beam2.rayleigh_range*100:.2f} cm")
print("All assertions passed ✓")

---
## Exercise 3 — `OpticalElement` hierarchy

In [ ]:
class OpticalElement:
    def matrix(self): raise NotImplementedError
    def propagate(self, ray): return self.matrix() @ ray
    def __matmul__(self, other):
        if isinstance(other, OpticalElement):
            return _MatrixElement(self.matrix() @ other.matrix())
        return NotImplemented
    def __repr__(self): return f"{self.__class__.__name__}()"

class _MatrixElement(OpticalElement):
    def __init__(self, M): self._M = M
    def matrix(self): return self._M

class FreeSpace(OpticalElement):
    def __init__(self, d):
        if d < 0: raise ValueError("d must be non-negative")
        self.d = d
    def matrix(self): return np.array([[1.0, self.d], [0.0, 1.0]])
    def __repr__(self): return f"FreeSpace(d={self.d*1e3:.1f} mm)"

class ThinLens(OpticalElement):
    def __init__(self, f):
        if f == 0: raise ValueError("f cannot be zero")
        self.f = f
    def matrix(self): return np.array([[1.0, 0.0], [-1.0/self.f, 1.0]])
    def __repr__(self): return f"ThinLens(f={self.f*1e3:.1f} mm)"

class Mirror(OpticalElement):
    def __init__(self, R): self.R = R
    def matrix(self): return np.array([[1.0, 0.0], [-2.0/self.R, 1.0]])
    def __repr__(self): return f"Mirror(R={self.R*1e3:.1f} mm)"


f1, f2 = 50e-3, 250e-3
expander = FreeSpace(f1) @ ThinLens(f1) @ FreeSpace(f1+f2) @ ThinLens(f2) @ FreeSpace(f2)
w0_in   = 1e-3
ray_in  = np.array([w0_in, 0.0])
ray_out = expander.propagate(ray_in)

assert isinstance(Mirror(0.3), OpticalElement)
assert isinstance(expander, OpticalElement)
assert np.isclose(np.abs(ray_out[0]), f2/f1*w0_in, rtol=0.01)
assert np.isclose(ray_out[1], 0.0, atol=1e-6)

print(f"Input  y={ray_in[0]*1e3:.1f} mm, θ={ray_in[1]*1e3:.2f} mrad")
print(f"Output y={ray_out[0]*1e3:.2f} mm, θ={ray_out[1]*1e6:.2f} µrad")
print(f"Magnification: {abs(ray_out[0]/ray_in[0]):.2f}×")
print("All assertions passed ✓")

---
## Exercise 4 — `PulseSequence` container

In [ ]:
class GaussianPulse:
    c = 3e8
    def __init__(self, tau, wavelength, energy):
        if tau<=0: raise ValueError("tau>0")
        if wavelength<=0: raise ValueError("wavelength>0")
        if energy<0: raise ValueError("energy>=0")
        self.tau=tau; self.wavelength=wavelength; self.energy=energy
    def __repr__(self): return f"GaussianPulse(τ={self.tau*1e15:.0f} fs, λ={self.wavelength*1e9:.0f} nm, E={self.energy*1e6:.1f} µJ)"
    def __eq__(self, other):
        if not isinstance(other, GaussianPulse): return NotImplemented
        return np.isclose(self.tau,other.tau) and np.isclose(self.wavelength,other.wavelength) and np.isclose(self.energy,other.energy)
    def __lt__(self, other): return self.energy < other.energy


class PulseSequence:
    def __init__(self, pulses=None): self._pulses = list(pulses) if pulses else []
    def __len__(self): return len(self._pulses)
    def __getitem__(self, idx): return self._pulses[idx]
    def __iter__(self): return iter(self._pulses)
    def __contains__(self, item): return item in self._pulses
    def __repr__(self): return f"PulseSequence({len(self)} pulses)"
    def __add__(self, other):
        if isinstance(other, PulseSequence): return PulseSequence(self._pulses + other._pulses)
        if isinstance(other, GaussianPulse): return PulseSequence(self._pulses + [other])
        return NotImplemented
    def total_energy(self): return sum(p.energy for p in self._pulses)
    def sort_by_energy(self, descending=False):
        return PulseSequence(sorted(self._pulses, key=lambda p: p.energy, reverse=descending))
    def filter_by_wavelength(self, lmin, lmax):
        return PulseSequence([p for p in self._pulses if lmin <= p.wavelength <= lmax])


pulses = [
    GaussianPulse(100e-15, 515e-9,  10e-6),
    GaussianPulse(150e-15, 800e-9,  50e-6),
    GaussianPulse(200e-15, 1030e-9, 80e-6),
    GaussianPulse(300e-15, 1310e-9, 35e-6),
    GaussianPulse(500e-15, 1550e-9, 20e-6),
]
seq = PulseSequence(pulses)
sorted_seq = seq.sort_by_energy(descending=True)
nir_seq    = seq.filter_by_wavelength(800e-9, 2000e-9)

assert len(seq)==5; assert pulses[0] in seq
assert sorted_seq[0].energy >= sorted_seq[1].energy
assert all(p.wavelength>=800e-9 for p in nir_seq)
assert np.isclose(seq.total_energy(), sum(p.energy for p in pulses))
assert sum(1 for _ in seq)==5
seq2 = seq + GaussianPulse(100e-15, 700e-9, 5e-6); assert len(seq2)==6

print(f"Sorted: {[p.energy*1e6 for p in sorted_seq]} µJ")
print(f"NIR: {nir_seq}")
print(f"Total energy: {seq.total_energy()*1e6:.1f} µJ")
print("All assertions passed ✓")

---
## Exercise 5 — `BeamParameters` dataclass

In [ ]:
@dataclass(frozen=True)
class BeamParameters:
    wavelength : float
    w0         : float
    M2         : float = 1.0
    z          : float = 0.0

    def __post_init__(self):
        if self.wavelength <= 0: raise ValueError("wavelength must be positive")
        if self.w0 <= 0:         raise ValueError("w0 must be positive")
        if self.M2 < 1.0:        raise ValueError(f"M² must be ≥ 1, got {self.M2}")

    @property
    def rayleigh_range(self): return np.pi * self.w0**2 / (self.M2 * self.wavelength)

    @property
    def w_at_z(self): return self.w0 * np.sqrt(1 + (self.z/self.rayleigh_range)**2)

    def peak_intensity(self, power): return 2 * power / (np.pi * self.w0**2)


class ShotRecord(NamedTuple):
    shot_id    : int
    energy_uJ  : float
    duration_fs: float
    beam       : BeamParameters


beam_params = BeamParameters(wavelength=800e-9, w0=400e-6, M2=1.1)
rng = np.random.default_rng(9)
records = [
    ShotRecord(i+1, float(rng.normal(50,2)), float(rng.normal(100,3)), beam_params)
    for i in range(5)
]
best = max(records, key=lambda r: r.energy_uJ)
beam_dict = {beam_params: 'main beam'}

assert beam_params.rayleigh_range > 0
assert np.isclose(beam_params.w_at_z, beam_params.w0)
assert hash(beam_params) is not None
assert beam_dict[beam_params] == 'main beam'
try: beam_params.w0 = 1e-3; assert False
except: pass
try: BeamParameters(800e-9, -1e-6); assert False
except ValueError: pass
sid, e, dur, b = records[0]; assert isinstance(b, BeamParameters)

print(beam_params)
print(f"zR = {beam_params.rayleigh_range*100:.2f} cm")
print(f"Best shot: #{best.shot_id} @ {best.energy_uJ:.2f} µJ")
print("All assertions passed ✓")

---
## Exercise 6 — `LaserSystem` by composition

In [ ]:
class GaussianPulse:
    def __init__(self, tau, wavelength, energy):
        self.tau=tau; self.wavelength=wavelength; self.energy=energy
    def __repr__(self): return f"GaussianPulse(τ={self.tau*1e15:.0f} fs, λ={self.wavelength*1e9:.0f} nm)"


class LaserSystem:
    def __init__(self, source, elements=None):
        self.source   = source
        self.elements = list(elements) if elements is not None else []

    def propagate_ray(self, ray):
        r = ray.copy()
        for el in self.elements:
            r = el.propagate(r)
        return r

    def add_element(self, element): self.elements.append(element)

    def remove_last(self): return self.elements.pop()

    def output_beam_radius(self, w_in):
        return abs(self.propagate_ray(np.array([w_in, 0.0]))[0])

    def __repr__(self):
        lines = [f"LaserSystem:", f"  source  : {self.source}"]
        for i, el in enumerate(self.elements):
            lines.append(f"  elem[{i}] : {el}")
        return '\n'.join(lines)


source = GaussianPulse(100e-15, 800e-9, 50e-6)
sys    = LaserSystem(source, [FreeSpace(0.1), ThinLens(0.1), FreeSpace(0.2)])
w_in   = 2e-3
w_out  = sys.output_beam_radius(w_in)

assert isinstance(sys.source, GaussianPulse)
assert len(sys.elements)==3
assert w_out != w_in
sys.add_element(FreeSpace(0.05)); assert len(sys.elements)==4
removed = sys.remove_last(); assert len(sys.elements)==3
assert isinstance(removed, FreeSpace)

print(sys)
print(f"\nInput  beam: {w_in*1e3:.2f} mm")
print(f"Output beam: {w_out*1e3:.2f} mm")
print("All assertions passed ✓")

---
## Exercise 7 — Full pulse characterisation pipeline

In [ ]:
class GaussianPulse:
    c = 3e8
    def __init__(self, tau, wavelength, energy):
        if tau<=0: raise ValueError("tau>0")
        self._tau=tau; self._wavelength=wavelength; self._energy=energy
    @property
    def tau(self): return self._tau
    @property
    def fwhm(self): return 2*np.sqrt(np.log(2))*self.tau
    @property
    def bandwidth(self): return np.sqrt(np.log(2))/(np.pi*self.tau)
    def intensity(self, t): return np.exp(-t**2/self.tau**2)
    def spectrum(self, t):
        N=len(t); dt=t[1]-t[0]
        E_f=np.fft.fftshift(np.fft.fft(np.sqrt(self.intensity(t))))
        return np.fft.fftshift(np.fft.fftfreq(N,d=dt)), np.abs(E_f)**2
    def __repr__(self): return f"GaussianPulse(τ={self.tau*1e15:.1f} fs)"

class ChirpedPulse(GaussianPulse):
    def __init__(self, tau, wavelength, energy, gdd):
        super().__init__(tau, wavelength, energy)
        self._gdd = float(gdd)
    @property
    def gdd(self): return self._gdd
    @property
    def tau_chirped(self): return self.tau*np.sqrt(1+(self.gdd/self.tau**2)**2)
    def intensity(self, t): return np.exp(-t**2/self.tau_chirped**2)
    def spectrum(self, t):
        N=len(t); dt=t[1]-t[0]
        freq=np.fft.fftshift(np.fft.fftfreq(N,d=dt))
        ω=2*np.pi*freq
        E_f=np.fft.fftshift(np.fft.fft(np.sqrt(np.exp(-t**2/self.tau**2))))*np.exp(0.5j*self.gdd*ω**2)
        return freq, np.abs(E_f)**2
    def __repr__(self): return f"ChirpedPulse(τ={self.tau*1e15:.1f} fs, GDD={self.gdd*1e30:.0f} fs²)"


class PulseAnalyser:
    def __init__(self, pulse, t):
        self.pulse = pulse
        self.t     = t
        self.I_t   = None; self.freq = None; self.S = None
        self.fwhm_t = None; self.fwhm_f = None; self.tbp = None

    @staticmethod
    def _fwhm(x, y):
        yn = y/y.max(); above = yn>=0.5
        return x[above].max()-x[above].min() if above.any() else np.nan

    def run(self):
        self.I_t          = self.pulse.intensity(self.t)
        self.freq, self.S = self.pulse.spectrum(self.t)
        self.fwhm_t       = self._fwhm(self.t, self.I_t)
        self.fwhm_f       = self._fwhm(self.freq, self.S)
        self.tbp          = self.fwhm_t * self.fwhm_f
        return self

    def plot(self, title=''):
        fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(10, 3.5))
        ax_t.plot(self.t*1e15, self.I_t, lw=2, color='C0')
        ax_t.set_xlabel('Time (fs)'); ax_t.set_ylabel('Intensity (norm.)')
        ax_t.set_title(f'{title} — time domain')
        ax_t.set_xlim(-1500, 1500)
        ax_f.plot(self.freq/1e12, self.S/self.S.max(), lw=2, color='C1')
        ax_f.set_xlabel('Frequency (THz)'); ax_f.set_ylabel('PSD (norm.)')
        ax_f.set_title(f'{title} — spectrum')
        ax_f.set_xlim(-10, 10)
        fig.suptitle(f'TBP = {self.tbp:.4f}', y=1.02)
        fig.tight_layout(); plt.show()

    def save(self, path):
        with h5py.File(path, 'w') as f:
            grp = f.create_group('pulse')
            grp.create_dataset('t',    data=self.t)
            grp.create_dataset('I_t',  data=self.I_t)
            grp.create_dataset('freq', data=self.freq)
            grp.create_dataset('S',    data=self.S)
            grp.attrs['fwhm_t_fs'] = self.fwhm_t * 1e15
            grp.attrs['fwhm_f_THz'] = self.fwhm_f / 1e12
            grp.attrs['tbp']        = self.tbp
            grp.attrs['pulse_repr'] = repr(self.pulse)


N_t=4096; dt_t=5e-15
t_ax = (np.arange(N_t)-N_t//2)*dt_t

p_TL = GaussianPulse(80e-15, 800e-9, 50e-6)
p_ch = ChirpedPulse( 80e-15, 800e-9, 50e-6, gdd=1000e-30)

an_TL = PulseAnalyser(p_TL, t_ax).run()
an_ch = PulseAnalyser(p_ch, t_ax).run()

an_TL.plot('Transform-limited')
an_ch.plot('Chirped (GDD=1000 fs²)')

with tempfile.TemporaryDirectory() as tmp:
    an_TL.save(os.path.join(tmp, 'TL.h5'))
    an_ch.save(os.path.join(tmp, 'chirped.h5'))
    print("HDF5 written")

assert hasattr(an_TL,'tbp') and hasattr(an_TL,'fwhm_t')
assert an_ch.tbp > an_TL.tbp
assert an_ch.fwhm_t > an_TL.fwhm_t

TBP_limit = 2*np.log(2)/np.pi
print(f"TL     TBP = {an_TL.tbp:.4f}  (transform limit: {TBP_limit:.4f})")
print(f"Chirped TBP = {an_ch.tbp:.4f}")
print("All assertions passed ✓")